# Ejemplo de Proyecto Integrador Completo: Prediccion de Riesgo Academico en FACES

**Modulo 04** | **Referencia** | **Proyecto completo ejecutable**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-04-proyectos-integradores/templates/ejemplo_proyecto_completo.ipynb)

---

## Seccion 1: Formulacion del Problema

### Contexto

La Facultad de Ciencias Economicas y Sociales (FACES) de la Universidad Catolica Andres Bello (UCAB) enfrenta un
desafio comun a muchas instituciones de educacion superior: la **desercion y el bajo rendimiento academico**.
Cuando un estudiante acumula un promedio inferior a 10 puntos, se considera en situacion de **riesgo academico**,
lo que puede derivar en reprobacion masiva, extension de la carrera o abandono definitivo.

Actualmente, la identificacion de estos estudiantes se realiza de manera **reactiva**: se detectan
cuando ya han reprobado varias materias. Un enfoque basado en datos permitiria una **intervencion temprana**,
canalizando recursos de tutoria y acompanamiento hacia quienes mas lo necesitan.

### Pregunta de investigacion

> **Que factores predicen que un estudiante este en riesgo academico (promedio < 10) en FACES?**

### Por que importa

- La intervencion temprana puede **reducir las tasas de desercion** entre un 15% y 25% segun la literatura.
- Permite **optimizar recursos limitados** de tutoria, dirigiendolos a quienes mas los necesitan.
- Genera **evidencia cuantitativa** para la toma de decisiones institucionales.
- Empodera al docente como **agente de deteccion temprana** en su propia catedra.

### Variables de interes

| Variable | Tipo | Descripcion | Rol |
|----------|------|-------------|-----|
| `promedio` | Continua | Promedio ponderado del estudiante (0-20) | Base para la variable objetivo |
| `en_riesgo` | Binaria | 1 si promedio < 10, 0 si no | **Variable objetivo** |
| `semestre` | Discreta | Semestre cursado (1-10) | Predictora |
| `edad` | Discreta | Edad del estudiante | Predictora |
| `asistencia_pct` | Continua | Porcentaje de asistencia (0-100) | Predictora |
| `materias_inscritas` | Discreta | Materias inscritas en el periodo | Predictora |
| `materias_aprobadas` | Discreta | Materias aprobadas en el periodo | Predictora |
| `materias_reprobadas` | Discreta | Materias reprobadas en el periodo | Predictora |
| `beca` | Binaria | Si el estudiante tiene beca | Predictora |
| `trabaja` | Binaria | Si el estudiante trabaja | Predictora |
| `carrera` | Categorica | Carrera que cursa | Predictora |
| `genero` | Categorica | Genero del estudiante (M/F) | Predictora |

---

## Seccion 2: Obtencion y Preparacion de Datos

In [ ]:
# ============================================================
# Celda 1: Importar librerias
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Modelos y evaluacion
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import LabelEncoder

# Configuracion global de visualizacion
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

print('Librerias cargadas correctamente')

In [ ]:
# ============================================================
# Celda 2: Cargar el dataset
# ============================================================
df = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')

print(f'Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nPrimeras 5 filas:')
df.head()

In [ ]:
# ============================================================
# Celda 3: Explorar la estructura del dataset
# ============================================================
print('Informacion del dataset:')
print('=' * 55)
df.info()

print('\n\nEstadisticas descriptivas de variables numericas:')
df.describe().round(2)

In [ ]:
# ============================================================
# Celda 4: Verificar valores faltantes
# ============================================================
nulos = df.isnull().sum()
print('Valores faltantes por columna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else 'No hay valores faltantes en el dataset.')

# Verificar duplicados
duplicados = df.duplicated(subset='estudiante_id').sum()
print(f'\nRegistros duplicados por estudiante_id: {duplicados}')

In [ ]:
# ============================================================
# Celda 5: Crear la variable objetivo y preparar features
# ============================================================

# Variable objetivo: en_riesgo = 1 si promedio < 10
df['en_riesgo'] = (df['promedio'] < 10).astype(int)

# Convertir columnas booleanas a enteros para el modelado
df['beca_num'] = df['beca'].astype(int)
df['trabaja_num'] = df['trabaja'].astype(int)

# Codificar genero: M=1, F=0
df['genero_num'] = (df['genero'] == 'M').astype(int)

# Codificar carrera con LabelEncoder
le_carrera = LabelEncoder()
df['carrera_cod'] = le_carrera.fit_transform(df['carrera'])

print('Variables creadas:')
print(f'  en_riesgo    : 1 si promedio < 10, 0 si no')
print(f'  beca_num     : 1 si tiene beca, 0 si no')
print(f'  trabaja_num  : 1 si trabaja, 0 si no')
print(f'  genero_num   : 1 si masculino, 0 si femenino')
print(f'  carrera_cod  : codificacion numerica de la carrera')
print(f'\nMapeo de carreras:')
for i, nombre in enumerate(le_carrera.classes_):
    print(f'    {i} = {nombre}')

In [ ]:
# ============================================================
# Celda 6: Verificar el balance de clases
# ============================================================
conteo = df['en_riesgo'].value_counts()
porcentaje = df['en_riesgo'].value_counts(normalize=True) * 100

print('Balance de la variable objetivo:')
print(f'  Sin riesgo (0): {conteo[0]:,} estudiantes ({porcentaje[0]:.1f}%)')
print(f'  En riesgo  (1): {conteo[1]:,} estudiantes ({porcentaje[1]:.1f}%)')
print(f'\n  Ratio: {conteo[0]/conteo[1]:.1f} : 1')

if porcentaje.min() < 20:
    print('\n  Nota: Las clases estan desbalanceadas. Esto puede afectar el modelo.')
    print('  Se usara stratify en train_test_split para mantener las proporciones.')
else:
    print('\n  Las clases estan razonablemente balanceadas.')

---

## Seccion 3: Analisis Exploratorio de Datos (EDA)

In [ ]:
# ============================================================
# Celda 7: Distribucion del promedio academico
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df['promedio'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=10, color='red', linestyle='--', linewidth=2, label='Umbral de riesgo (promedio = 10)')

# Sombrear la zona de riesgo
ax.axvspan(df['promedio'].min(), 10, alpha=0.1, color='red', label='Zona de riesgo')

ax.set_xlabel('Promedio academico')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribucion del Promedio Academico de los Estudiantes')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Promedio general: {df["promedio"].mean():.2f}')
print(f'Mediana: {df["promedio"].median():.2f}')
print(f'Desviacion estandar: {df["promedio"].std():.2f}')
print(f'Estudiantes en riesgo (promedio < 10): {(df["promedio"] < 10).sum()} ({(df["promedio"] < 10).mean()*100:.1f}%)')

In [ ]:
# ============================================================
# Celda 8: Tasa de riesgo por carrera
# ============================================================
riesgo_carrera = df.groupby('carrera')['en_riesgo'].agg(['mean', 'sum', 'count'])
riesgo_carrera.columns = ['tasa_riesgo', 'total_en_riesgo', 'total_estudiantes']
riesgo_carrera = riesgo_carrera.sort_values('tasa_riesgo', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.RdYlGn_r(riesgo_carrera['tasa_riesgo'] / riesgo_carrera['tasa_riesgo'].max())
barras = ax.barh(riesgo_carrera.index, riesgo_carrera['tasa_riesgo'] * 100, color=colores, edgecolor='white')

# Agregar etiquetas con porcentaje y conteo
for i, (tasa, total, n) in enumerate(zip(riesgo_carrera['tasa_riesgo'], 
                                          riesgo_carrera['total_en_riesgo'],
                                          riesgo_carrera['total_estudiantes'])):
    ax.text(tasa * 100 + 0.5, i, f'{tasa*100:.1f}% ({int(total)}/{int(n)})', va='center', fontsize=11)

ax.set_xlabel('Tasa de riesgo (%)')
ax.set_title('Tasa de Riesgo Academico por Carrera')
ax.set_xlim(0, riesgo_carrera['tasa_riesgo'].max() * 100 + 10)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Celda 9: Mapa de correlacion de variables numericas
# ============================================================
variables_numericas = ['semestre', 'edad', 'promedio', 'materias_inscritas',
                       'materias_aprobadas', 'materias_reprobadas', 
                       'asistencia_pct', 'beca_num', 'trabaja_num', 'en_riesgo']

corr_matrix = df[variables_numericas].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mascara = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mascara, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlacion'})
ax.set_title('Mapa de Correlacion entre Variables Numericas', fontsize=14)
plt.tight_layout()
plt.show()

# Correlaciones con la variable objetivo
corr_objetivo = corr_matrix['en_riesgo'].drop('en_riesgo').sort_values()
print('Correlacion de cada variable con en_riesgo:')
for var, corr in corr_objetivo.items():
    signo = '+' if corr > 0 else ''
    print(f'  {var:25s}: {signo}{corr:.3f}')

In [ ]:
# ============================================================
# Celda 10: Box plots de variables clave segun estado de riesgo
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribucion de Variables Clave segun Estado de Riesgo', fontsize=15, y=1.02)

variables_box = ['asistencia_pct', 'materias_aprobadas', 'materias_reprobadas',
                 'semestre', 'edad', 'materias_inscritas']
titulos_box = ['Asistencia (%)', 'Materias aprobadas', 'Materias reprobadas',
               'Semestre', 'Edad', 'Materias inscritas']

etiquetas_riesgo = {0: 'Sin riesgo', 1: 'En riesgo'}
df['estado_riesgo'] = df['en_riesgo'].map(etiquetas_riesgo)

for i, (var, titulo) in enumerate(zip(variables_box, titulos_box)):
    ax = axes[i // 3][i % 3]
    sns.boxplot(data=df, x='estado_riesgo', y=var, ax=ax,
                palette={'Sin riesgo': '#2ecc71', 'En riesgo': '#e74c3c'},
                order=['Sin riesgo', 'En riesgo'])
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(titulo)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Celda 11: Tabulacion cruzada de beca y riesgo
# ============================================================
tabla_beca = pd.crosstab(df['beca'], df['en_riesgo'], margins=True, margins_name='Total')
tabla_beca.columns = ['Sin riesgo', 'En riesgo', 'Total']
tabla_beca.index = ['Sin beca', 'Con beca', 'Total']

print('Tabulacion cruzada: Beca x Riesgo Academico')
print('=' * 45)
print(tabla_beca)

# Tasas de riesgo
tasa_sin_beca = df[df['beca'] == False]['en_riesgo'].mean() * 100
tasa_con_beca = df[df['beca'] == True]['en_riesgo'].mean() * 100

print(f'\nTasa de riesgo SIN beca: {tasa_sin_beca:.1f}%')
print(f'Tasa de riesgo CON beca: {tasa_con_beca:.1f}%')

In [ ]:
# ============================================================
# Celda 12: Tabulacion cruzada de trabaja y riesgo
# ============================================================
tabla_trabaja = pd.crosstab(df['trabaja'], df['en_riesgo'], margins=True, margins_name='Total')
tabla_trabaja.columns = ['Sin riesgo', 'En riesgo', 'Total']
tabla_trabaja.index = ['No trabaja', 'Trabaja', 'Total']

print('Tabulacion cruzada: Trabaja x Riesgo Academico')
print('=' * 50)
print(tabla_trabaja)

tasa_no_trabaja = df[df['trabaja'] == False]['en_riesgo'].mean() * 100
tasa_trabaja = df[df['trabaja'] == True]['en_riesgo'].mean() * 100

print(f'\nTasa de riesgo NO trabaja: {tasa_no_trabaja:.1f}%')
print(f'Tasa de riesgo SI trabaja: {tasa_trabaja:.1f}%')

In [ ]:
# ============================================================
# Celda 13: Riesgo por genero y semestre
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Riesgo por genero
riesgo_genero = df.groupby('genero')['en_riesgo'].mean() * 100
axes[0].bar(riesgo_genero.index, riesgo_genero.values, 
            color=['#3498db', '#e74c3c'], edgecolor='white', width=0.5)
for i, (g, v) in enumerate(zip(riesgo_genero.index, riesgo_genero.values)):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Tasa de riesgo (%)')
axes[0].set_title('Tasa de Riesgo por Genero')
axes[0].set_ylim(0, riesgo_genero.max() + 5)

# Riesgo por semestre
riesgo_semestre = df.groupby('semestre')['en_riesgo'].mean() * 100
axes[1].plot(riesgo_semestre.index, riesgo_semestre.values, 'o-', 
             color='steelblue', linewidth=2, markersize=8)
axes[1].set_xlabel('Semestre')
axes[1].set_ylabel('Tasa de riesgo (%)')
axes[1].set_title('Tasa de Riesgo por Semestre')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.show()

### Hallazgos clave del EDA

1. **Materias reprobadas** es la variable con mayor correlacion con el riesgo academico: a mas materias reprobadas, mayor probabilidad de estar en riesgo.
2. **Materias aprobadas y asistencia** muestran una relacion inversa: estudiantes con mas materias aprobadas y mayor asistencia tienden a estar fuera de riesgo.
3. **Las diferencias entre carreras** existen pero son moderadas. No hay una carrera con tasas dramaticamente diferentes.
4. **La beca y el trabajo** tienen efectos que vale la pena cuantificar con el modelo, aunque las diferencias en tasa de riesgo no son extremas en el analisis bivariado.

---

## Seccion 4: Modelado Predictivo

In [ ]:
# ============================================================
# Celda 14: Seleccion de features y division train/test
# ============================================================

# Seleccionamos features que estarian disponibles al inicio del semestre
# Nota: NO incluimos promedio (se uso para crear la variable objetivo)
# ni materias_aprobadas/reprobadas (son resultado, no predictor temprano)
features = ['semestre', 'edad', 'asistencia_pct', 'materias_inscritas',
             'beca_num', 'trabaja_num', 'genero_num', 'carrera_cod']

X = df[features]
y = df['en_riesgo']

# Division 80% entrenamiento, 20% prueba
# stratify=y mantiene la proporcion de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Division de datos:')
print(f'  Entrenamiento: {len(X_train)} registros ({len(X_train)/len(X)*100:.0f}%)')
print(f'  Prueba:        {len(X_test)} registros ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nProporcion en riesgo:')
print(f'  Entrenamiento: {y_train.mean()*100:.1f}%')
print(f'  Prueba:        {y_test.mean()*100:.1f}%')
print(f'\nFeatures seleccionadas ({len(features)}):')
for f in features:
    print(f'  - {f}')

In [ ]:
# ============================================================
# Celda 15: Modelo 1 -- Regresion Logistica
# ============================================================
# La regresion logistica es un buen punto de partida:
# - Simple e interpretable
# - Sirve como linea base
# - Los coeficientes indican la direccion del efecto

modelo_lr = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr.fit(X_train, y_train)

y_pred_lr = modelo_lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print('Modelo 1: Regresion Logistica')
print('=' * 40)
print(f'Accuracy en test: {acc_lr:.4f} ({acc_lr*100:.1f}%)')
print(f'\nReporte de clasificacion:')
print(classification_report(y_test, y_pred_lr, target_names=['Sin riesgo', 'En riesgo']))

In [ ]:
# ============================================================
# Celda 16: Modelo 2 -- Arbol de Decision
# ============================================================
# El arbol de decision es muy visual y facil de explicar:
# - Genera reglas tipo "si asistencia < 75% Y trabaja = 1 → en riesgo"
# - Limitamos la profundidad para evitar sobreajuste

modelo_dt = DecisionTreeClassifier(max_depth=5, random_state=42, min_samples_leaf=20)
modelo_dt.fit(X_train, y_train)

y_pred_dt = modelo_dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print('Modelo 2: Arbol de Decision (profundidad max = 5)')
print('=' * 50)
print(f'Accuracy en test: {acc_dt:.4f} ({acc_dt*100:.1f}%)')
print(f'\nReporte de clasificacion:')
print(classification_report(y_test, y_pred_dt, target_names=['Sin riesgo', 'En riesgo']))

In [ ]:
# ============================================================
# Celda 17: Visualizacion del arbol de decision
# ============================================================
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(modelo_dt, feature_names=features, class_names=['Sin riesgo', 'En riesgo'],
          filled=True, rounded=True, ax=ax, fontsize=8, max_depth=3,
          impurity=False, proportion=True)
ax.set_title('Arbol de Decision (primeros 3 niveles)', fontsize=14)
plt.tight_layout()
plt.show()

print('El arbol muestra las reglas de decision que el modelo aprendio.')
print('Cada nodo indica: la condicion, la proporcion de cada clase y la clase predicha.')

In [ ]:
# ============================================================
# Celda 18: Modelo 3 -- Random Forest
# ============================================================
# Random Forest combina muchos arboles para obtener mejor precision:
# - Mas robusto que un solo arbol
# - Menos propenso al sobreajuste
# - Proporciona importancia de features

modelo_rf = RandomForestClassifier(
    n_estimators=100,    # 100 arboles
    max_depth=8,         # profundidad maxima
    min_samples_leaf=10, # minimo de muestras por hoja
    random_state=42,
    n_jobs=-1            # usar todos los nucleos
)
modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print('Modelo 3: Random Forest (100 arboles, profundidad max = 8)')
print('=' * 55)
print(f'Accuracy en test: {acc_rf:.4f} ({acc_rf*100:.1f}%)')
print(f'\nReporte de clasificacion:')
print(classification_report(y_test, y_pred_rf, target_names=['Sin riesgo', 'En riesgo']))

In [ ]:
# ============================================================
# Celda 19: Validacion cruzada para comparar modelos
# ============================================================
# La validacion cruzada es mas robusta que una sola division train/test.
# Usamos 5 folds estratificados.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'Arbol de Decision': DecisionTreeClassifier(max_depth=5, random_state=42, min_samples_leaf=20),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, 
                                             min_samples_leaf=10, random_state=42, n_jobs=-1)
}

resultados_cv = {}
print('Validacion Cruzada (5 folds) — Accuracy')
print('=' * 55)

for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X, y, cv=cv, scoring='accuracy')
    resultados_cv[nombre] = scores
    print(f'{nombre:25s}: {scores.mean():.4f} (+/- {scores.std():.4f})')
    print(f'  Scores por fold: {[f"{s:.4f}" for s in scores]}')

# Identificar mejor modelo
mejor = max(resultados_cv, key=lambda k: resultados_cv[k].mean())
print(f'\nMejor modelo: {mejor}')

In [ ]:
# ============================================================
# Celda 20: Visualizacion comparativa de modelos
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

nombres = list(resultados_cv.keys())
medias = [resultados_cv[n].mean() for n in nombres]
desviaciones = [resultados_cv[n].std() for n in nombres]

colores_modelos = ['#3498db', '#2ecc71', '#e74c3c']
barras = ax.bar(nombres, medias, yerr=desviaciones, capsize=8,
                color=colores_modelos, edgecolor='white', alpha=0.85)

for barra, media in zip(barras, medias):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.005,
            f'{media:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Accuracy (validacion cruzada 5 folds)')
ax.set_title('Comparacion de Modelos de Clasificacion')
ax.set_ylim(0, 1.0)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Azar (50%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Celda 21: Seleccionar mejor modelo y mostrar matriz de confusion
# ============================================================
# Usamos Random Forest como modelo final (mejor en validacion cruzada)
modelo_final = modelo_rf
y_pred_final = y_pred_rf
nombre_final = 'Random Forest'

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Sin riesgo', 'En riesgo'],
            yticklabels=['Sin riesgo', 'En riesgo'],
            annot_kws={'size': 20}, ax=ax)
ax.set_xlabel('Prediccion del modelo', fontsize=13)
ax.set_ylabel('Valor real', fontsize=13)
ax.set_title(f'Matriz de Confusion — {nombre_final}', fontsize=14)
plt.tight_layout()
plt.show()

# Interpretar la matriz
tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos negativos (correctamente sin riesgo): {tn}')
print(f'Falsos positivos (falsas alarmas):               {fp}')
print(f'Falsos negativos (en riesgo no detectados):      {fn}')
print(f'Verdaderos positivos (en riesgo detectados):     {tp}')

In [ ]:
# ============================================================
# Celda 22: Reporte de clasificacion detallado con interpretacion
# ============================================================
print(f'Reporte de Clasificacion Final — {nombre_final}')
print('=' * 55)
print(classification_report(y_test, y_pred_final, 
                            target_names=['Sin riesgo', 'En riesgo']))

# Metricas individuales
prec = precision_score(y_test, y_pred_final)
rec = recall_score(y_test, y_pred_final)
f1 = f1_score(y_test, y_pred_final)

print('Interpretacion practica:')
print(f'  Precision = {prec:.2f}')
print(f'    De cada 100 estudiantes que el modelo senala como "en riesgo",')
print(f'    {prec*100:.0f} realmente lo estan. El resto son falsas alarmas.')
print(f'\n  Recall = {rec:.2f}')
print(f'    De cada 100 estudiantes realmente en riesgo,')
print(f'    el modelo detecta a {rec*100:.0f}. Los demas pasan desapercibidos.')
print(f'\n  F1 Score = {f1:.2f}')
print(f'    Balance entre precision y recall. Util cuando ambas metricas importan.')

---

## Seccion 5: Interpretacion de Resultados

In [ ]:
# ============================================================
# Celda 23: Importancia de features (Random Forest)
# ============================================================
importancias = pd.DataFrame({
    'Feature': features,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colores_imp = plt.cm.viridis(np.linspace(0.2, 0.8, len(importancias)))
ax.barh(importancias['Feature'], importancias['Importancia'], color=colores_imp)
ax.set_xlabel('Importancia (reduccion media de impureza)')
ax.set_title('Importancia de Variables en el Modelo Random Forest', fontsize=14)

# Agregar porcentajes
for i, (feat, imp) in enumerate(zip(importancias['Feature'], importancias['Importancia'])):
    ax.text(imp + 0.002, i, f'{imp*100:.1f}%', va='center', fontsize=11)

plt.tight_layout()
plt.show()

print('Las 3 variables mas importantes para predecir riesgo academico:')
top3 = importancias.tail(3).iloc[::-1]
for i, (_, row) in enumerate(top3.iterrows(), 1):
    print(f'  {i}. {row["Feature"]:25s} ({row["Importancia"]*100:.1f}%)')

In [ ]:
# ============================================================
# Celda 24: Coeficientes de la regresion logistica (interpretabilidad)
# ============================================================
# Aunque Random Forest fue el mejor modelo, los coeficientes de la
# regresion logistica son mas faciles de interpretar directamente.

coef_df = pd.DataFrame({
    'Variable': features,
    'Coeficiente': modelo_lr.coef_[0]
}).sort_values('Coeficiente')

fig, ax = plt.subplots(figsize=(10, 6))
colores_coef = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coeficiente']]
ax.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colores_coef)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Coeficiente')
ax.set_title('Coeficientes de la Regresion Logistica', fontsize=14)
plt.tight_layout()
plt.show()

print('Interpretacion de coeficientes:')
print('  Rojo  = coeficiente positivo = AUMENTA la probabilidad de riesgo')
print('  Verde = coeficiente negativo = DISMINUYE la probabilidad de riesgo')
print(f'\nIntercepto: {modelo_lr.intercept_[0]:.4f}')

In [ ]:
# ============================================================
# Celda 25: SHAP values (interpretabilidad avanzada)
# ============================================================
# SHAP (SHapley Additive exPlanations) muestra el impacto de cada
# feature en la prediccion de cada estudiante individual.

try:
    import shap
    
    # Crear el explicador para Random Forest
    explainer = shap.TreeExplainer(modelo_rf)
    shap_values = explainer.shap_values(X_test)
    
    # Summary plot: impacto global de cada feature
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values[1], 
                      X_test, feature_names=features, show=False)
    plt.title('SHAP Summary Plot — Impacto de cada variable en la prediccion', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    print('Cada punto es un estudiante. La posicion horizontal indica')
    print('cuanto esa variable empuja la prediccion hacia riesgo (derecha)')
    print('o fuera de riesgo (izquierda). El color indica el valor de la variable.')
    
except ImportError:
    print('La libreria SHAP no esta instalada.')
    print('Para instalarla: pip install shap')
    print('\nSin SHAP, la importancia de features del Random Forest y los')
    print('coeficientes de la regresion logistica siguen siendo interpretables.')
except Exception as e:
    print(f'No se pudo generar el grafico SHAP: {e}')
    print('Continuamos con los metodos de interpretacion anteriores.')

### Que nos dice el modelo

Los tres modelos coinciden en los factores mas relevantes para predecir el riesgo academico:

1. **Asistencia** es el factor mas importante. Estudiantes con baja asistencia tienen una probabilidad significativamente mayor de estar en riesgo. Esto es consistente con la literatura sobre abandono universitario.

2. **Numero de materias inscritas** tiene un efecto relevante. Puede reflejar la carga academica del estudiante y su capacidad de gestion del tiempo.

3. **Variables demograficas** (edad, semestre) aportan informacion complementaria, aunque con menor poder predictivo individual.

4. **Beca y trabajo** tienen efectos detectables pero menores en magnitud que las variables academicas directas.

### Limitaciones y advertencias

Es fundamental reconocer las limitaciones de este analisis:

1. **Datos sinteticos**: Este dataset es generado para fines educativos. Un modelo en produccion requeriria datos reales validados por la institucion.

2. **Variables omitidas**: Factores importantes como situacion socioeconomica, salud mental, motivacion, calidad docente y ambiente familiar no estan disponibles en el dataset.

3. **Causalidad vs. correlacion**: El modelo identifica **asociaciones**, no causas. Que la asistencia prediga el riesgo no significa que obligar a asistir resuelva el problema; la baja asistencia puede ser sintoma de otras dificultades.

4. **Sesgo potencial**: Si los datos historicos reflejan sesgos sistematicos (por genero, origen, etc.), el modelo los reproducira. Es necesaria una auditoria de equidad antes de usar el modelo para decisiones que afecten a personas.

5. **Generalizacion**: Un modelo entrenado con datos de una facultad y periodo especifico puede no aplicar a otros contextos sin recalibracion.

### Recomendaciones para el coordinador academico

Basandose en los hallazgos del modelo, se sugieren las siguientes acciones:

1. **Sistema de alerta temprana basado en asistencia**: Implementar un monitoreo de asistencia en las primeras 4 semanas del semestre. Estudiantes con asistencia inferior al 75% deberian recibir una notificacion automatica y ser contactados por su tutor.

2. **Asesoria en carga academica**: Ofrecer orientacion personalizada a estudiantes que inscriban un numero inusualmente alto o bajo de materias, ya que ambos extremos estan asociados con mayor riesgo.

3. **Programas de acompanamiento diferenciado**: Utilizar las predicciones del modelo para priorizar la asignacion de tutores y mentores a los estudiantes con mayor probabilidad de riesgo.

4. **Evaluacion periodica del modelo**: Recalibrar el modelo cada semestre con datos actualizados para mantener su precision y detectar cambios en los patrones de riesgo.

5. **Uso etico**: Las predicciones deben usarse para **apoyar**, nunca para penalizar. El modelo es una herramienta de deteccion, no un juicio sobre el estudiante.

---

## Seccion 6: Presentacion de Resultados

In [ ]:
# ============================================================
# Celda 26: Dashboard resumen (figura 2x2 con los 4 graficos clave)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Dashboard: Prediccion de Riesgo Academico en FACES', 
             fontsize=16, fontweight='bold', y=1.02)

# --- Panel 1: Distribucion del promedio ---
ax1 = axes[0, 0]
ax1.hist(df['promedio'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(x=10, color='red', linestyle='--', linewidth=2)
ax1.axvspan(df['promedio'].min(), 10, alpha=0.1, color='red')
ax1.set_xlabel('Promedio')
ax1.set_ylabel('Frecuencia')
ax1.set_title('A. Distribucion del Promedio Academico')
ax1.text(5, ax1.get_ylim()[1] * 0.9, f'{(df["promedio"]<10).mean()*100:.1f}%\nen riesgo', 
         fontsize=12, color='red', fontweight='bold', ha='center')

# --- Panel 2: Importancia de features ---
ax2 = axes[0, 1]
imp_sorted = importancias.copy()
colores_dash = plt.cm.viridis(np.linspace(0.2, 0.8, len(imp_sorted)))
ax2.barh(imp_sorted['Feature'], imp_sorted['Importancia'], color=colores_dash)
ax2.set_xlabel('Importancia')
ax2.set_title('B. Variables mas Importantes (Random Forest)')

# --- Panel 3: Matriz de confusion ---
ax3 = axes[1, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Sin riesgo', 'En riesgo'],
            yticklabels=['Sin riesgo', 'En riesgo'],
            annot_kws={'size': 18}, ax=ax3)
ax3.set_xlabel('Prediccion')
ax3.set_ylabel('Real')
ax3.set_title(f'C. Matriz de Confusion ({nombre_final})')

# --- Panel 4: Comparacion de modelos ---
ax4 = axes[1, 1]
nombres_cortos = ['Logistica', 'Arbol', 'Random Forest']
medias_cv = [resultados_cv[n].mean() for n in resultados_cv]
desv_cv = [resultados_cv[n].std() for n in resultados_cv]
ax4.bar(nombres_cortos, medias_cv, yerr=desv_cv, capsize=8,
        color=colores_modelos, edgecolor='white', alpha=0.85)
for i, (m, d) in enumerate(zip(medias_cv, desv_cv)):
    ax4.text(i, m + d + 0.01, f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')
ax4.set_ylabel('Accuracy (CV 5 folds)')
ax4.set_title('D. Comparacion de Modelos')
ax4.set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Celda 27: Perfil de estudiantes en riesgo vs. sin riesgo
# ============================================================
perfil = df.groupby('en_riesgo')[['asistencia_pct', 'materias_inscritas', 
                                    'edad', 'semestre']].mean().round(2)
perfil.index = ['Sin riesgo', 'En riesgo']
perfil.columns = ['Asistencia (%)', 'Materias inscritas', 'Edad', 'Semestre']

print('Perfil promedio de estudiantes segun estado de riesgo:')
print('=' * 60)
print(perfil.to_string())

# Agregar tasas de beca y trabajo
print(f'\nTasa de beca:')
print(f'  Sin riesgo: {df[df["en_riesgo"]==0]["beca"].mean()*100:.1f}%')
print(f'  En riesgo:  {df[df["en_riesgo"]==1]["beca"].mean()*100:.1f}%')
print(f'\nTasa de trabajo:')
print(f'  Sin riesgo: {df[df["en_riesgo"]==0]["trabaja"].mean()*100:.1f}%')
print(f'  En riesgo:  {df[df["en_riesgo"]==1]["trabaja"].mean()*100:.1f}%')

In [ ]:
# ============================================================
# Celda 28: Simulacion de aplicacion practica
# ============================================================
# Simular como se usaria el modelo con nuevos estudiantes

# Tomar 5 estudiantes del conjunto de prueba como ejemplo
muestra = X_test.head(5).copy()
predicciones = modelo_rf.predict(muestra)
probabilidades = modelo_rf.predict_proba(muestra)[:, 1]

# Construir tabla de resultados
resultado_simulacion = muestra.copy()
resultado_simulacion['Prob. riesgo'] = [f'{p*100:.1f}%' for p in probabilidades]
resultado_simulacion['Prediccion'] = ['EN RIESGO' if p == 1 else 'Sin riesgo' for p in predicciones]
resultado_simulacion['Real'] = ['EN RIESGO' if r == 1 else 'Sin riesgo' for r in y_test.head(5).values]

print('Simulacion: Prediccion para 5 estudiantes nuevos')
print('=' * 80)
print(resultado_simulacion[['asistencia_pct', 'materias_inscritas', 'beca_num', 
                             'trabaja_num', 'Prob. riesgo', 'Prediccion', 'Real']].to_string(index=False))
print('\nEste tipo de tabla es lo que el coordinador veria al inicio del semestre')
print('para priorizar el acompanamiento a los estudiantes con mayor probabilidad de riesgo.')

### Resumen ejecutivo

Se desarrollo un modelo predictivo para identificar estudiantes en riesgo academico (promedio < 10) en la Facultad de Ciencias Economicas y Sociales. Utilizando datos de 3,000 estudiantes y 8 variables predictoras, se compararon tres algoritmos de clasificacion: Regresion Logistica, Arbol de Decision y Random Forest. El modelo Random Forest obtuvo el mejor desempeno en validacion cruzada. Los factores mas importantes para predecir el riesgo fueron la asistencia a clases, el numero de materias inscritas y la edad del estudiante. El modelo puede servir como base para un sistema de alerta temprana que permita al coordinador academico intervenir de manera proactiva antes de que el estudiante acumule resultados negativos irreversibles.

### Como usaria esto en mi catedra

Este proyecto integrador demuestra como un docente de FACES puede aplicar las herramientas aprendidas en el programa de formacion:

1. **En la ensenanza**: Usar este tipo de proyecto como caso practico en materias de estadistica, investigacion de operaciones o administracion. Los estudiantes pueden replicar el flujo completo con datos de su propia disciplina.

2. **En la gestion academica**: Proponer al departamento un piloto de sistema de alerta temprana basado en este modelo. Con datos reales anonimizados, se podria implementar una version funcional.

3. **En la investigacion**: Este analisis puede ser el punto de partida para un articulo sobre factores de riesgo academico en universidades venezolanas, un area con poca literatura cuantitativa.

4. **En la evaluacion**: Utilizar proyectos similares como instrumento de evaluacion en materias cuantitativas, donde el estudiante demuestre competencias de analisis de datos de principio a fin.

### Referencias

1. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer. Capitulos 3, 4 y 8.

2. Geron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly. Capitulos 3 y 7.

3. Scikit-learn developers. (2024). *User Guide: Supervised Learning*. https://scikit-learn.org/stable/supervised_learning.html

4. Tinto, V. (1975). Dropout from Higher Education: A Theoretical Synthesis of Recent Research. *Review of Educational Research*, 45(1), 89-125.

5. Lundberg, S. M., & Lee, S. I. (2017). A Unified Approach to Interpreting Model Predictions. *Advances in Neural Information Processing Systems*, 30.

6. Provost, F., & Fawcett, T. (2013). *Data Science for Business*. O'Reilly. Capitulo 8: Visualizing Model Performance.

---

*Este notebook fue creado como ejemplo de referencia para el programa de Formacion Docente en Business Intelligence de FACES. Es un proyecto completo y ejecutable que sigue las 6 secciones del template de proyecto integrador del Modulo 04.*